# FX Pass 优化

## 1. 功能简介

`npugraph_ex` 集成了 PyTorch 原生 Pattern 能力，可按照预定义的替换规则匹配 FX Graph 中的算子组合，并使用等价的融合算子完成替换。该过程无需修改原始模型代码，可减少部分场景中的任务下发和调度开销。

`pattern_fusion_pass` 为布尔类型，默认值为 `True`，统一控制 TorchAir 内置及用户注册的算子融合 Pass。设置为 `False` 时，两类融合规则都会关闭。不同版本支持的规则可能变化，具体能力应以当前版本的 [TorchAir 文档](https://gitcode.com/Ascend/torchair/blob/master/docs/zh/npugraph_ex/basic/pattern_fusion_pass.md)为准。

### 1.1 当前内置融合规则

<table style="float: left;">
  <tr><th align="left">典型替换规则</th><th align="left">对应融合算子</th></tr>
  <tr><td align="left"><code>npu_add_rms_norm</code> 输出作为 <code>npu_dynamic_quant</code> 输入</td><td align="left"><code>npu_add_rms_norm_dynamic_quant</code></td></tr>
  <tr><td align="left"><code>npu_add_rms_norm</code> 输出经过 view/类型转换</td><td align="left"><code>npu_add_rms_norm_cast</code></td></tr>
  <tr><td align="left">三维 matmul 与指定维度 transpose 组合</td><td align="left"><code>npu_transpose_batchmatmul</code></td></tr>
  <tr><td align="left"><code>npu_add_rms_norm</code> 输出作为 <code>npu_quantize</code> 输入</td><td align="left"><code>npu_add_rms_norm_quant</code></td></tr>
</table>
<div style="clear: both;"></div>

> **注意事项：** 只有完整满足替换规则时才会发生融合。融合算子的输出必须被正常使用，且融合后消失的中间结果不能被其他位置引用；<code>npu_transpose_batchmatmul</code> 等规则还可能因输入 Shape 变化而失效，并触发 FX Graph 重新编译。

**图 1**  FX Pass 在编译流程中的位置

<img src="./images/fx_pass_pipeline.svg" width="900">

Pattern 融合 Pass 基于已有 Aten IR，在 FX Graph 编译后、NPUGraph Capture 前执行。图变换完成后，需要依次验证融合是否命中、数值精度和实际性能，不能仅根据配置项已开启就判断优化生效。

In [ ]:
import torch
import torch_npu

assert torch.npu.is_available(), "请在已安装 CANN 和 torch_npu 的昇腾环境中运行"
torch.manual_seed(0)
print("PyTorch:", torch.__version__)
print("torch_npu:", torch_npu.__version__)
print("Device:", torch.npu.get_device_name(0))
# 构造简单逐元素子图，用于演示 pattern_fusion_pass 的配置方式。
class PointwiseModel(torch.nn.Module):
    def forward(self, x, residual):
        hidden = torch.relu(x + residual)
        return hidden.to(torch.float32)


# 注意：该简化模型不对应官方内置规则，因此不保证产生融合算子。
model = PointwiseModel().npu()
compiled = torch.compile(
    model,
    backend="npugraph_ex",
    options={"pattern_fusion_pass": True},
    fullgraph=True,
    dynamic=False,
)
x = torch.randn(4, 16, dtype=torch.float16).npu()
residual = torch.randn(4, 16, dtype=torch.float16).npu()
# 首次执行后可结合 Debug Dump 查看优化后的 FX 图结构。
out = compiled(x, residual)
print(out.dtype, tuple(out.shape))


## 2. 自定义算子融合

当内置规则不能覆盖业务中的高频算子组合时，可以使用 [register_replacement](https://gitcode.com/Ascend/torchair/blob/master/docs/zh/npugraph_ex/api/npugraph_ex/register_replacement.md#register_replacement) 注册自定义融合规则。自定义规则同样由 `pattern_fusion_pass` 统一控制。

- search_fn 描述原始 FX 子图；
- replace_fn 描述替换后的融合算子；
- example_inputs 用于初次 trace；
- extra_check 可对 shape、设备或其他条件做二次校验。

下面是接口骨架。若替换实现使用自定义融合算子，还需完成[自定义算子入图](https://gitcode.com/Ascend/torchair/blob/master/docs/zh/custom_op_graph/custom_op_graph.md)，并由使用者保证匹配条件、数据依赖和替换前后的数值语义正确。

In [ ]:
# search_fn 描述需要在 FX Graph 中匹配的原始子图。
def search_fn(x, y):
    return torch.add(x, y)


# replace_fn 描述匹配成功后的等价替换实现。
def replace_fn(x, y):
    # 使用目标 torch_npu 版本实际支持的融合算子替换这里。
    return torch.add(x, y)


# extra_check 可进一步限制 Shape、dtype 或设备等匹配条件。
def extra_check(match):
    return True


# torch.npu.npugraph_ex.register_replacement(
#     search_fn=search_fn,
#     replace_fn=replace_fn,
#     example_inputs=(
#         torch.empty(2, 8, device="npu"),
#         torch.empty(2, 8, device="npu"),
#     ),
#     extra_check=extra_check,
# )
print("先验证 search/replace 的数学等价性，再注册自定义 Pass")


## 3. 验证 Pass 是否生效

1. 参考[图编译 Debug 信息保存功能](https://gitcode.com/Ascend/torchair/blob/master/docs/zh/npugraph_ex/dfx/debug_save.md)，设置 `TORCH_COMPILE_DEBUG=1`；
2. 在 Debug 信息的 `npugraph_ex` 目录及模型 `forward` 子目录中查看 FX 图优化输出，确认目标融合算子是否出现；
3. 对比优化前后的节点、数据依赖和输出结构，并用 Eager 结果做精度基线；
4. 使用 Profiler 比较任务下发、内存搬运和稳定区间端到端耗时。

不要仅凭节点变少或配置项已开启判断收益。规则未满足时，不产生融合属于正常现象；自定义规则还必须自行保证替换前后的数学等价性，并通过实际 Profiling 评估收益。

## 4. 课后练习

### 一、单选题

（1）【单选题】FX Pass 在 TorchAir 编译流程中的典型位置是？
- A. Python 代码解释之前
- B. Dynamo 生成 FX Graph 之后、NPUGraph Capture 之前
- C. NPU Kernel 执行完成之后
- D. 模型保存到磁盘之后

（2）【单选题】关于 `pattern_fusion_pass` 的说法，正确的是？
- A. 默认值为 True，同时控制内置及用户注册的模式融合规则
- B. 默认值为 False，仅控制内置模式融合规则
- C. 用于控制输入地址是否固定
- D. 用于控制图内存池复用

（3）【单选题】使用 `register_replacement` 时，`search_fn` 的作用是？
- A. 描述待匹配的原始 FX 子图
- B. 执行 NPU Profiling
- C. 创建图内存池
- D. 强制重新 Capture

（4）【单选题】`example_inputs` 在自定义替换规则中的主要用途是？
- A. 保存全部模型输出
- B. 用于初次 trace
- C. 自动规避所有图中断
- D. 指定 NPU 的数量

（5）【单选题】自定义 `replace_fn` 最基本的正确性要求是？
- A. 必须比原子图包含更多节点
- B. 与被替换子图保持数学等价并通过精度验证
- C. 只能调用 Python 内置函数
- D. 不允许使用目标版本支持的融合算子

（6）【单选题】验证 FX Pass 是否实际改变了图结构，最直接的方法是？
- A. 查看优化前后的 FX 图
- B. 修改随机种子
- C. 删除模型参数
- D. 只运行一次模型

### 二、多选题

（7）【多选题】FX Pass 可以带来哪些能力？
- A. 不修改原始模型代码即可做图变换
- B. 对匹配模式执行算子融合
- C. 执行原地化或自定义子图替换
- D. 自动免除所有精度验证

（8）【多选题】验证一个 FX Pass 是否有效，应包括哪些步骤？
- A. 对比优化前后的 FX 图
- B. 检查融合算子符号、节点数和数据依赖
- C. 与 Eager 结果比较数值精度，并用 Profiler 评估性能
- D. 只要节点变少就直接上线

（9）【多选题】`register_replacement` 的组成或校验项包括哪些？
- A. search_fn
- B. replace_fn
- C. example_inputs
- D. extra_check

（10）【多选题】实现自定义 FX Pass 时，正确的原则有哪些？
- A. 以目标版本支持的融合算子和规则为准
- B. 保证替换前后的数据依赖和数值语义正确
- C. 自定义融合算子还需要完成自定义算子入图
- D. 不需要进行实际性能测试

**运行以下代码单元查看参考答案与解析。**


In [ ]:
import os
answer_path = "answer/03.05_answer.txt"
if os.path.exists(answer_path):
    with open(answer_path, "r", encoding="utf-8") as f:
        print(f.read())
else:
    print("答案文件未找到，请检查 answer 目录。")
